In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np

# Generate synthetic raw customer dataset with issues
np.random.seed(42)
n_train = 100
n_test = 40

def generate_customer_df(n):
    df = pd.DataFrame({
        'CustomerID': np.arange(1, n+1),
        'Age': np.random.choice(
            np.append(np.random.randint(18, 70, n-5), [120, -5, np.nan, 200, 999]), n, replace=False
        ),
        'Gender': np.random.choice(['Male', 'Female', 'Other', 'unknown', np.nan], n),
        'Income': np.random.choice(
            np.append(np.random.normal(50000, 15000, n-3), [np.nan, 1e6, -1000]), n, replace=False
        ),
        'Country': np.random.choice(['USA', 'Canada', 'UK', 'India', 'unknown', np.nan], n),
        'PurchaseAmount': np.random.choice(
            np.append(np.random.normal(200, 80, n-2), [np.nan, 10000]), n, replace=False
        )
    })
    # Introduce some string numbers and mixed types
    df.loc[df.sample(frac=0.05).index, 'Age'] = df['Age'].sample(frac=0.05).astype(str)
    df.loc[df.sample(frac=0.05).index, 'Income'] = df['Income'].sample(frac=0.05).astype(str)
    return df

train_raw = generate_customer_df(n_train)
test_raw = generate_customer_df(n_test)

# Copy for processing
train = train_raw.copy()
test = test_raw.copy()

# Data type correction
for df in [train, test]:
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df['Income'] = pd.to_numeric(df['Income'], errors='coerce')
    df['PurchaseAmount'] = pd.to_numeric(df['PurchaseAmount'], errors='coerce')
    df['Gender'] = df['Gender'].astype(str).str.strip().str.title()
    df['Country'] = df['Country'].astype(str).str.strip().str.title()

# Handle missing values: impute Age/Income/PurchaseAmount with median, categorical with mode
for df in [train, test]:
    for col in ['Age', 'Income', 'PurchaseAmount']:
        median = df[col].median()
        df[col] = df[col].fillna(median)
    for col in ['Gender', 'Country']:
        mode = df[col][df[col] != 'Nan'].mode()[0]
        df[col] = df[col].replace('Nan', np.nan)
        df[col] = df[col].fillna(mode)

# Handle outliers: cap Age between 18 and 100, Income between 0 and 200000, PurchaseAmount between 0 and 5000
for df in [train, test]:
    df['Age'] = df['Age'].clip(18, 100)
    df['Income'] = df['Income'].clip(0, 200000)
    df['PurchaseAmount'] = df['PurchaseAmount'].clip(0, 5000)

# Standardize categorical values
for df in [train, test]:
    df['Gender'] = df['Gender'].replace({'Unknown': 'Other'})
    df['Country'] = df['Country'].replace({'Unknown': 'Other'})

# Display summary of cleaned data
print("Train Data Summary:")
print(train.info())
print(train.describe(include='all'))

print("\nTest Data Summary:")
print(test.info())
print(test.describe(include='all'))

Train Data Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      100 non-null    int64  
 1   Age             100 non-null    float64
 2   Gender          100 non-null    object 
 3   Income          100 non-null    float64
 4   Country         100 non-null    object 
 5   PurchaseAmount  100 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 4.8+ KB
None
        CustomerID         Age Gender         Income Country  PurchaseAmount
count   100.000000  100.000000    100     100.000000     100      100.000000
unique         NaN         NaN      3            NaN       5             NaN
top            NaN         NaN  Other            NaN   Other             NaN
freq           NaN         NaN     47            NaN      38             NaN
mean     50.500000   45.120000    NaN   49998.794151     NaN     

In [3]:
# Compute descriptive statistics for numerical columns
num_cols = train.select_dtypes(include=[np.number]).columns.drop('CustomerID')
cat_cols = train.select_dtypes(include=['object']).columns

def describe_numerical(df, cols, name):
    print(f"\nNumerical Descriptive Stats for {name}:")
    desc = df[cols].agg(['count', 'mean', 'median', 'std']).T
    print(desc)
    return desc

def frequency_tables(df, cols, name):
    print(f"\nFrequency Tables for {name}:")
    for col in cols:
        print(f"\n{col} value counts:")
        print(df[col].value_counts())
    return

# Train set
train_num_desc = describe_numerical(train, num_cols, "Train Set")
frequency_tables(train, cat_cols, "Train Set")

# Test set
test_num_desc = describe_numerical(test, num_cols, "Test Set")
frequency_tables(test, cat_cols, "Test Set")


Numerical Descriptive Stats for Train Set:
                count          mean        median           std
Age             100.0     45.120000     43.000000     17.471639
Income          100.0  49998.794151  48083.977573  22651.992747
PurchaseAmount  100.0    253.240023    208.788806    486.366859

Frequency Tables for Train Set:

Gender value counts:
Gender
Other     47
Male      37
Female    16
Name: count, dtype: int64

Country value counts:
Country
Other     38
Usa       17
Canada    16
India     16
Uk        13
Name: count, dtype: int64

Numerical Descriptive Stats for Test Set:
                count          mean        median           std
Age              40.0     46.975000     44.000000     19.982669
Income           40.0  48926.022948  46084.591201  29322.622112
PurchaseAmount   40.0    355.591205    246.714255    756.537613

Frequency Tables for Test Set:

Gender value counts:
Gender
Male      18
Other     13
Female     9
Name: count, dtype: int64

Country value counts:
Cou

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for reproducibility
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Numerical columns for visualization
num_cols = ['Age', 'Income', 'PurchaseAmount']
cat_cols = ['Gender', 'Country']

# 1. Histograms for numerical columns
for col in num_cols:
    plt.figure()
    sns.histplot(train[col], kde=True, bins=20, color='skyblue')
    plt.title(f'Train Set: Histogram of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(f'train_hist_{col}.png')
    plt.close()

# 2. Box plots for numerical columns
for col in num_cols:
    plt.figure()
    sns.boxplot(x=train[col], color='lightgreen')
    plt.title(f'Train Set: Boxplot of {col}')
    plt.xlabel(col)
    plt.tight_layout()
    plt.savefig(f'train_box_{col}.png')
    plt.close()

# 3. Bar charts for categorical columns
for col in cat_cols:
    plt.figure()
    sns.countplot(x=train[col], palette='pastel', order=train[col].value_counts().index)
    plt.title(f'Train Set: {col} Distribution')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig(f'train_bar_{col}.png')
    plt.close()

# Concise report
report = """
# Customer Dataset Descriptive Analysis Report

## Data Cleaning
- Missing values in numerical columns were imputed with the median; categorical columns with the mode.
- Outliers were capped: Age (18-100), Income (0-200,000), PurchaseAmount (0-5,000).
- Categorical values were standardized.

## Key Metrics (Train Set)
- **Age**: Mean = {:.2f}, Median = {:.2f}, Std = {:.2f}
- **Income**: Mean = {:.2f}, Median = {:.2f}, Std = {:.2f}
- **PurchaseAmount**: Mean = {:.2f}, Median = {:.2f}, Std = {:.2f}

## Categorical Distributions
- **Gender**: {}
- **Country**: {}

## Visualizations
- Histograms: `train_hist_Age.png`, `train_hist_Income.png`, `train_hist_PurchaseAmount.png`
- Boxplots: `train_box_Age.png`, `train_box_Income.png`, `train_box_PurchaseAmount.png`
- Bar charts: `train_bar_Gender.png`, `train_bar_Country.png`

""".format(
    train['Age'].mean(), train['Age'].median(), train['Age'].std(),
    train['Income'].mean(), train['Income'].median(), train['Income'].std(),
    train['PurchaseAmount'].mean(), train['PurchaseAmount'].median(), train['PurchaseAmount'].std(),
    train['Gender'].value_counts().to_dict(),
    train['Country'].value_counts().to_dict()
)

# Save report
with open("customer_descriptive_report.md", "w") as f:
    f.write(report)

print("Visualizations saved as PNG files and report generated as 'customer_descriptive_report.md'.")

Visualizations saved as PNG files and report generated as 'customer_descriptive_report.md'.
